# Konu 5 — Dil modelleri nasıl çalışır?

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Isınma ve Adım 1–5 **anahtarsız** çalışır. Adım 6–7 `anahtar.txt` ister.

## Isınma — cümleyi tamamla

Sınıfa sorduk: *"Sessiz bir çalışma ..."* cümlesinin sonunu ne getirirsin?
Altı kişinin cevabı aşağıda. En sık iki cevap hangisi? `Counter` Konu 1'den tanıdık.

In [ ]:
from collections import Counter

cevaplar = ["kafesi", "kafesi", "yeri", "odası", "kafesi", "yeri"]
sayac = Counter(cevaplar)
print(sayac.most_common(2))

Bir sonraki kelimeyi tahmin etmenin en basit yolu: **en sık söyleneni seç.**
Bu konunun bütün fikri bu hücrede. Dil modeli de bunu yapar, yalnızca milyarlarca metinle.

## Adım 1 — Model metni nasıl görür: belirteçler

Önce tarayıcıda: `tiktokenizer.vercel.app` sitesine bir cümle yaz, parçalara nasıl bölündüğüne bak.
Şimdi aynısını kodla. Model metni **kelime kelime** değil, **parça parça** okur.
Bu parçalara **belirteç** (token) denir.

İlk çalıştırmada `tiktoken` internetten bir sözlük dosyası indirir; birkaç saniye sürebilir.

In [ ]:
import tiktoken

kodlayici = tiktoken.get_encoding("o200k_base")
parcalar = kodlayici.encode("Sessiz bir çalışma kafesi")
print(len(parcalar))
print(parcalar)

Model kelimeleri değil, bu **sayıları** görür. Her sayı bir parça. Parçaları geri çevirelim.
`decode` bir **liste** ister; o yüzden tek parçayı köşeli parantez içinde veriyoruz:

In [ ]:
for parca in parcalar:
    print(kodlayici.decode([parca]))

Dört kelime, altı parça. `bir` ve `çalışma` tek parça, ama `Sessiz` ve `kafesi` ikiye bölündü.
Parçanın başındaki boşluk da parçaya dahil.

Aynı anlamda İngilizce bir cümle kaç parça?

In [ ]:
ingilizce = kodlayici.encode("A quiet cafe for working")
print(len(ingilizce))

Beş kelime, beş parça. Türkçe ekler yüzünden daha çok parçaya bölünür. Uzun bir Türkçe kelime deneyelim:

In [ ]:
uzun = kodlayici.encode("Kütüphanedekilerden misiniz?")
print(len(uzun))
for parca in uzun:
    print(kodlayici.decode([parca]))

İki kelime, on bir parça. Model için Türkçe metin daha "uzun": aynı sözü söylemek için daha çok
parça harcar. Modelin bir seferde okuyabildiği parça sayısı sınırlıdır (**bağlam penceresi**);
Türkçe metin bu pencereyi daha çabuk doldurur.

## Adım 2 — Hangi kelimeden sonra ne geliyor?

Konu 1'deki 30 kafe yorumu (dosyanın aynısı bu konunun `veri` klasöründe).
Konu 1'deki temizlik satırlarının aynısı:

In [ ]:
with open("veri/kafe-yorumlari.txt", encoding="utf-8") as dosya:
    metin = dosya.read()

temiz = metin.replace("I", "ı")
temiz = temiz.replace("İ", "i")
temiz = temiz.lower()
for isaret in ".,;:!?'()-":
    temiz = temiz.replace(isaret, " ")

kelimeler = temiz.split()
print(len(kelimeler))

Şimdi "biraz" kelimesinden **hemen sonra** gelen kelimeleri topluyoruz.
Döngü her kelimede bir önceki kelimeye bakar: önceki "biraz" ise bu kelimeyi listeye ekler.
Son satır, bu kelimeyi bir sonraki tur için "önceki" olarak hatırlar.

In [ ]:
hedef = "biraz"
sonra_gelenler = []
onceki = ""
for kelime in kelimeler:
    if onceki == hedef:
        sonra_gelenler.append(kelime)
    onceki = kelime

print(sonra_gelenler)

Müşteriler "biraz"dan sonra ne yazmış? Hepsi ne tür kelimeler?

Şimdi "çok". Aynı hücre, tek fark ilk satır:

In [ ]:
hedef = "çok"
sonra_gelenler = []
onceki = ""
for kelime in kelimeler:
    if onceki == hedef:
        sonra_gelenler.append(kelime)
    onceki = kelime

print(sonra_gelenler)

Hangisi en sık? Isınmadaki `Counter`:

In [ ]:
sayac = Counter(sonra_gelenler)
print(sayac.most_common(5))

## Adım 3 — Sayıdan olasılığa

"çok"tan sonra 5 kelime gelmiş; "güzel" bunların 2'si. Yani "güzel"in olasılığı 2 / 5.
Her kelimenin adedini toplama bölüyoruz:

In [ ]:
toplam = len(sonra_gelenler)
etiketler = []
olasiliklar = []
for kelime, adet in sayac.most_common(5):
    olasilik = adet / toplam
    etiketler.append(kelime)
    olasiliklar.append(olasilik)
    print(kelime, olasilik)

Olasılıkları topla: kaç ediyor? Konu 1 Adım 8'deki çubuk grafiğin aynısı:

In [ ]:
import matplotlib.pyplot as plt

plt.bar(etiketler, olasiliklar, color="#4a6fa5")
plt.title("'çok' kelimesinden sonra ne geliyor?")
plt.ylabel("Olasılık")
plt.show()

Bir dil modeli de her adımda böyle bir dağılım hesaplar: "bundan sonra hangi parça, hangi olasılıkla?"
Farkı, bunu 30 yorumdan değil milyarlarca metinden öğrenmiş olması.

## Adım 4 — Metin üret: hep en olasıyı seç

Adım 2'nin satırlarını bir fonksiyona koyuyoruz. Fonksiyon bir kelime alır,
ondan sonra en sık gelen kelimeyi geri verir.
`most_common(1)` tek elemanlı bir liste verir; `en_sik[0]` o elemanı, yani (kelime, adet) ikilisini alır.

In [ ]:
def sonraki_kelime(hedef):
    sonra_gelenler = []
    onceki = ""
    for kelime in kelimeler:
        if onceki == hedef:
            sonra_gelenler.append(kelime)
        onceki = kelime
    sayac = Counter(sonra_gelenler)
    en_sik = sayac.most_common(1)
    secilen, adet = en_sik[0]
    return secilen

In [ ]:
print(sonraki_kelime("çok"))

Metin üretmek = bu tahmini tekrarlamak. "kahve"den başlıyoruz; her turda son kelimenin ardından
en olası kelimeyi ekliyoruz. `range(8)`: 8 kez tekrarla.

In [ ]:
kelime = "kahve"
cumle = kelime
for tur in range(8):
    kelime = sonraki_kelime(kelime)
    cumle = cumle + " " + kelime

print(cumle)

Cümle bir şey anlatıyor mu? Hücreyi bir kez daha çalıştır: değişti mi?

Hep en olasıyı seçince her seferinde **aynı** cümle çıkar. Model bir anlam kurmuyor; yalnızca sayıyor.

## Adım 5 — Zar at

Şimdi en sıkını değil, **zarla** seçelim. `random.choice` listeden rastgele bir eleman seçer.
"çok"tan sonra gelenler listesinde "güzel" iki kez var; o yüzden seçilme şansı da iki kat.
Hücreyi birkaç kez çalıştır:

In [ ]:
import random

print(sonra_gelenler)
print(random.choice(sonra_gelenler))

Adım 4'teki fonksiyonun aynısı; yalnızca son iki satır farklı:

In [ ]:
def sonraki_kelime_zarla(hedef):
    sonra_gelenler = []
    onceki = ""
    for kelime in kelimeler:
        if onceki == hedef:
            sonra_gelenler.append(kelime)
        onceki = kelime
    secilen = random.choice(sonra_gelenler)
    return secilen

Aynı üretim, zarla. Hücreyi üç kez çalıştır, cümleleri karşılaştır:

In [ ]:
kelime = "kahve"
cumle = kelime
for tur in range(8):
    kelime = sonraki_kelime_zarla(kelime)
    cumle = cumle + " " + kelime

print(cumle)

Her çalıştırmada başka bir cümle. Ara sıra `IndexError: Cannot choose from an empty sequence`
görebilirsin: zar "kahveli"ye geldi. O kelime dosyanın **son** kelimesi; ondan sonra hiçbir şey
gelmiyor. Hücreyi yeniden çalıştır.

Gerçek modellerde bu ayarın adı **sıcaklık**:

- Sıcaklık düşük ≈ hep en olasıyı seç (Adım 4): her seferinde aynı
- Sıcaklık yüksek ≈ zar at (Adım 5): her seferinde farklı

## Adım 6 — Gerçek model ve sıcaklık

Bu adımdan sonrası `anahtar.txt` ister. Konu 2'nin Adım 1 hücresinin aynısı:

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print("Anahtar okundu")

Adres ve başlık Konu 2'deki gibi. `modele_sor` fonksiyonu da Konu 2'deki; tek fark gövdeye
eklenen `"temperature"` (sıcaklık) alanı.

In [ ]:
import requests

MODEL = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}

In [ ]:
def modele_sor(soru, sicaklik):
    govde = {"prompt": soru, "temperature": sicaklik}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    yanit = cevap.json()
    sonuc = yanit["result"]
    return sonuc["response"]

Aynı soruyu üç sıcaklıkta soruyoruz.
**Çalıştırmadan önce tahmin et:** hangisi en "garip" slogan olacak?

In [ ]:
soru = "Sessiz bir çalışma kafesi için tek bir kısa slogan yaz. Yalnızca sloganı yaz."

for sicaklik in [0.1, 0.7, 1.5]:
    metin = modele_sor(soru, sicaklik)
    print(sicaklik, metin)

Düşük sıcaklıkta iki kez soralım. İki cevap aynı mı?

In [ ]:
ilk = modele_sor(soru, 0.1)
ikinci = modele_sor(soru, 0.1)
print(ilk)
print(ikinci)

Şimdi yüksek sıcaklıkta iki kez:

In [ ]:
ilk = modele_sor(soru, 1.5)
ikinci = modele_sor(soru, 1.5)
print(ilk)
print(ikinci)

Adım 4 ile Adım 5'i hatırla: hangisi hangisine benziyor?

## Adım 7 — Slogan panosu

Her sıcaklıkta 3 slogan üretip hepsini başlıklarıyla tek bir dosyaya yazıyoruz.
`"w"` "yazmak için aç", `"\n"` yeni satır. 9 istek gidiyor; hücre biraz sürer.

In [ ]:
with open("pano.txt", "w", encoding="utf-8") as dosya:
    for sicaklik in [0.1, 0.7, 1.5]:
        dosya.write("Sıcaklık " + str(sicaklik) + "\n")
        for tur in range(3):
            metin = modele_sor(soru, sicaklik)
            dosya.write(metin + "\n")
        dosya.write("\n")

print("pano.txt yazıldı")

Klasörde `pano.txt`'yi aç, üç bölümü yan yana oku.

- Kurumsal bir tabela için hangi sıcaklık?
- Fikir fırtınası için hangisi?
- 1.5'teki sloganlardan kullanılabilecek olan var mı?

## Bonus

**1.** Kendi adını ve soyadını belirteçlere ayır. Kaç parça? Yanındakinin adı kaç parça?

In [ ]:
parcalar = kodlayici.encode("Ahmet Emre Aladağ")
print(len(parcalar))
for parca in parcalar:
    print(kodlayici.decode([parca]))

**2.** Adım 4'teki üretim hücresinde `"kahve"` yerine `"sessiz"` ya da `"priz"` yaz, yeniden çalıştır.
Dikkat: `"kahveli"` gibi dosyanın son kelimesiyle başlarsan `IndexError` alırsın; ondan sonra hiç kelime yok.